# High-Dimensional Spectral Analysis of GPT-2
### Layer Complexity Reduction & Denoised Synthetic Generation via Random Matrix Theory (RMT)

This notebook demonstrates:
1. **Weight Matrix Extraction**: Extracting Attention & MLP projections across GPT-2 transformer blocks.
2. **Empirical Correlation & ESD**: Computing $C = \frac{1}{N} W W^\top$ and eigenvalue spectrum.
3. **Marchenko-Pastur Fit**: Identifying bulk noise edge $\lambda_+$ vs informative signal outlier spikes.
4. **Layer Complexity Metrics**: Stable Rank, Effective Rank, and Spectral Entropy.
5. **Layer Pruning**: Removing redundant blocks and evaluating perplexity.
6. **Spectral Denoising**: Filtering intra-layer noise for synthetic data generation.

In [ ]:
import sys, os
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt

from src.models.gpt2_extractor import load_gpt2_model_and_tokenizer, extract_layer_weights, count_parameters
from src.spectral.rmt_analysis import analyze_matrix_spectrum, compute_correlation_matrix
from src.pruning.layer_reduction import compute_layer_importance_scores, select_layers_to_prune, prune_model_and_benchmark
from src.utils.visualizer import plot_esd_and_mp, plot_scree_and_powerlaw, plot_layer_metrics_profile

print(f"PyTorch Version: {torch.__version__}")

## 1. Load Pretrained GPT-2

In [ ]:
model, tokenizer = load_gpt2_model_and_tokenizer('gpt2', device='cpu')
params_info = count_parameters(model)
print(f"Loaded GPT-2 ({len(model.transformer.h)} layers) with {params_info['total_parameters']:,} parameters ({params_info['size_mb']:.1f} MB)")

## 2. Spectral Analysis of Layer 0 Attention Projection

In [ ]:
l0_weights = extract_layer_weights(model, layer_idx=0)
l0_attn_proj = l0_weights.attn_proj.numpy()

res = analyze_matrix_spectrum(l0_attn_proj, name='layer_0_attn_proj')

print(f"Matrix Shape           : {res.shape}")
print(f"Aspect Ratio Q         : {res.aspect_ratio_Q:.2f}")
print(f"MP Bulk Upper Edge     : {res.lambda_plus:.4f}")
print(f"Stable Rank            : {res.stable_rank:.2f}")
print(f"Effective Rank (erank) : {res.effective_rank:.2f}")
print(f"Signal Energy Ratio    : {res.signal_energy_ratio * 100:.2f}%")
print(f"Noise Energy Ratio     : {res.noise_energy_ratio * 100:.2f}%")

## 3. Visualize Empirical Spectral Density vs Marchenko-Pastur Fit

In [ ]:
plot_esd_and_mp(
    eigenvalues=res.eigenvalues,
    lambda_minus=res.lambda_minus,
    lambda_plus=res.lambda_plus,
    sigma_sq=res.sigma_sq,
    Q=res.aspect_ratio_Q,
    title='GPT-2 Layer 0 Attention Projection - ESD vs Marchenko-Pastur'
)
plt.show()

## 4. Layer-Wise Spectral Profile & Redundancy Ranking

In [ ]:
layer_scores = compute_layer_importance_scores(model)
plot_layer_metrics_profile(layer_scores)
plt.show()

## 5. Prune Least Informative Layers

In [ ]:
eval_texts = ["Random Matrix Theory provides rigorous tools for analyzing high-dimensional deep learning."]
benchmark = prune_model_and_benchmark(
    model=model,
    tokenizer=tokenizer,
    num_layers_to_prune=3,
    eval_texts=eval_texts,
    strategy='min_signal_energy'
)

print(f"Pruned Layers       : {benchmark['pruned_indices']}")
print(f"Parameter Reduction : {benchmark['param_reduction_pct']:.2f}%")
print(f"Baseline PPL        : {benchmark['original_perplexity']:.2f}")
print(f"Pruned PPL          : {benchmark['pruned_perplexity']:.2f}")